# 🧩 Prerequisites

In [ ]:
!pip install -q uv
!uv pip install langchain langgraph langchain-ollama sentence-transformers nltk

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Agentic Summarization Framework

In [ ]:
import nltk
import operator
import torch
from google.colab import userdata
from typing import List, Dict, TypedDict, Literal
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Ensure NLTK sentence tokenizer is available
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

# --- 1. STATE DEFINITION ---
class SummarizationState(TypedDict):
    source_text: str
    clinical_facts: List[str]
    current_summary: str
    summary_sentences: List[str]
    nli_matrix: List[List[float]]
    hallucinated_spans: List[Dict]
    loop_count: int

# --- 2. MODEL INITIALIZATION ---
# Generative LLM (Can be swapped for a local clinical LLM via Ollama/vLLM)
llm = ChatOllama(
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {'Authorization': 'Bearer ' + userdata.get('OLLAMA_API_KEY')}
    },
    model="gemma4:31b-cloud"
)
# llm = ChatGoogleGenerativeAI(model="gemma-4-31b-it", thinking_level="minimal")

# NLI Critic (Proxy-SummaC)
model_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, device_map="auto")
model.eval()

def nli_pipeline(premise, hypothesis):
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True
    ).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    return probs[model.config.label2id["entailment"]].item()


MAX_ITERATIONS = 3
ENTAILMENT_THRESHOLD = 0.65

In [ ]:
# nli_pipeline("The heart size is within normal limits.", "heart size and mediastinal contours are within normal limits.")
nli_pipeline("heart size and mediastinal contours are within normal limits.", "The heart size is within normal limits.")

In [ ]:
# --- NODE A: Atomic Fact Extractor ---
def extract_atomic_facts(state: SummarizationState):
    class ClinicalClaims(BaseModel):
        facts: List[str] = Field(description="List of atomic clinical claims or facts")

    parser = PydanticOutputParser(pydantic_object=ClinicalClaims)

    prompt = f"""
    You are an expert radiologist. Extract a strict list of atomic clinical claims from the following report.
    Prioritize anatomical locations, measurements, and explicit negations. Do NOT summarize, just list facts.

    Report: {state['source_text']}

    {parser.get_format_instructions()}
    """
    llm_with_structure = llm.with_structured_output(ClinicalClaims, method="json_mode")
    response = llm_with_structure.invoke([HumanMessage(content=prompt)])
    return {"clinical_facts": response.facts}

# --- NODE B: Clinical Draft Summarizer ---
def draft_summary(state: SummarizationState):
    facts_str = "\n".join([f"- {f}" for f in state['clinical_facts']])
    prompt = f"""
    Draft a clinical 'Impression' section based strictly on the following atomic facts.
    Do not add outside information. Also do not miss any information.

    Facts:
    {facts_str}

    Impression:
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"current_summary": response.text, "loop_count": 0}

# --- NODE C: Proxy-SummaC Evaluator ---
def proxy_summac_evaluator(state: SummarizationState):
    source_sentences = sent_tokenize(state['source_text'])
    summary_sentences = sent_tokenize(state['current_summary'])

    nli_matrix = []
    hallucinated_spans = []

    # Calculate M_{ij}
    for i, s_i in enumerate(summary_sentences):
        row_scores = []
        for d_j in source_sentences:
            # Premise = source doc (d_j), Hypothesis = summary sentence (s_i)
            ent_score = nli_pipeline(d_j, s_i)

            # Extract entailment probability (logic depends on specific model labels)
            entailment_score = ent_score
            row_scores.append(entailment_score)

        nli_matrix.append(row_scores)
        max_entailment = max(row_scores) if row_scores else 0

        # Check against threshold tau
        if max_entailment < ENTAILMENT_THRESHOLD:
            hallucinated_spans.append({
                "index": i,
                "sentence": s_i,
                "max_score": max_entailment
            })

    return {
        "summary_sentences": summary_sentences,
        "nli_matrix": nli_matrix,
        "hallucinated_spans": hallucinated_spans
    }

# --- NODE D: Targeted Revision Agent ---
def targeted_revision(state: SummarizationState):
    sentences = state['summary_sentences'].copy()

    for span in state['hallucinated_spans']:
        idx = span['index']
        bad_sentence = span['sentence']

        prompt = f"""
        The following sentence in a radiology impression is flagged as an extrinsic hallucination or lacks entailment:
        "{bad_sentence}"

        Source Report:
        {state['source_text']}

        Rewrite ONLY this flagged sentence to be strictly factual based on the Source Report.
        Convert compound sentence into separate sentences, if needed, to get higher entailment with the Source Report.
        If the sentence cannot be salvaged, return 'DROP_SENTENCE'. Do not output anything else.
        """
        response = llm.invoke([HumanMessage(content=prompt)]).text.strip()

        if "DROP_SENTENCE" in response:
            sentences[idx] = "" # Mark for removal
        else:
            sentences[idx] = response

    # Reconstruct summary
    new_summary = " ".join([s for s in sentences if s]).strip()
    return {"current_summary": new_summary, "loop_count": state['loop_count'] + 1}

# --- FALLBACK NODE: Force Prune ---
def force_prune(state: SummarizationState):
    sentences = state['summary_sentences'].copy()
    # Drop all stubbornly unentailed sentences
    for span in state['hallucinated_spans']:
        sentences[span['index']] = ""

    new_summary = " ".join([s for s in sentences if s]).strip()
    return {"current_summary": new_summary}

# --- FINAL POLISHER NODE ---
def final_polisher(state: SummarizationState):
    prompt = f"""
    The following clinical impression has been surgically edited for factuality.
    Smooth out any disjointed transitions to ensure it reads professionally.
    DO NOT add any new clinical facts or change the medical meaning.

    Below is an example showing a Draft Impression and Final Impression format:

    Draft Impression:
    Impression:
    - 3mm right upper lobe nodule, unchanged from one year ago. \
    - Mild degenerative changes in the thoracic spine. \
    - Clear lungs without focal consolidation, pleural effusion, or pneumothorax. Heart size is within normal limits. Mediastinal contours are unremarkable.

    Final Impression:
    1. Stable 3mm right upper lobe nodule.
    2. Mild thoracic spine degenerative changes.
    3. No focal consolidation, pleural effusion, or pneumothorax.
    4. Normal heart size and mediastinal contours.

    Now, give the Final Impression given the example format above:

    Draft Impression:
    {state['current_summary']}

    Final Impression:
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"current_summary": response.text}


def route_after_evaluation(state: SummarizationState) -> Literal["final_polisher", "targeted_revision", "force_prune"]:
    hallucinations = state.get("hallucinated_spans", [])
    current_loop = state.get("loop_count", 0)

    # Condition 1: Perfect Score
    if not hallucinations:
        return "final_polisher"

    # Condition 2: Hallucinations exist, but we have loops left
    if current_loop < MAX_ITERATIONS:
        return "targeted_revision"

    # Condition 3: Max iterations reached, force prune to guarantee SummaC
    return "force_prune"

In [ ]:
# Initialize Graph
workflow = StateGraph(SummarizationState)

# Add Nodes
workflow.add_node("extract_atomic_facts", extract_atomic_facts)
workflow.add_node("draft_summary", draft_summary)
workflow.add_node("proxy_summac_evaluator", proxy_summac_evaluator)
workflow.add_node("targeted_revision", targeted_revision)
workflow.add_node("force_prune", force_prune)
workflow.add_node("final_polisher", final_polisher)

# Define Linear Edges
workflow.add_edge(START, "extract_atomic_facts")
workflow.add_edge("extract_atomic_facts", "draft_summary")
workflow.add_edge("draft_summary", "proxy_summac_evaluator")

# Define Conditional Routing
workflow.add_conditional_edges(
    "proxy_summac_evaluator",
    route_after_evaluation,
    {
        "final_polisher": "final_polisher",
        "targeted_revision": "targeted_revision",
        "force_prune": "force_prune"
    }
)

# Close loops
workflow.add_edge("targeted_revision", "proxy_summac_evaluator") # Route back to evaluate the revision
workflow.add_edge("force_prune", "final_polisher")
workflow.add_edge("final_polisher", END)

# Setup a checkpointer
memory = MemorySaver()

# Compile
app = workflow.compile(checkpointer=memory)

In [ ]:
app

## Test

In [ ]:
sample_radiology_report = """
PA and lateral views of the chest provided. The heart size is within normal limits.
The mediastinal contours are unremarkable. The lungs are clear without focal consolidation,
pleural effusion, or pneumothorax. There are mild degenerative changes in the thoracic spine.
A 3mm nodule is noted in the right upper lobe, which is unchanged from the prior study dated one year ago.
"""

initial_state = {
    "source_text": sample_radiology_report,
    "clinical_facts": [],
    "current_summary": "",
    "summary_sentences": [],
    "nli_matrix": [],
    "hallucinated_spans": [],
    "loop_count": 0
}

thread = {"configurable": {"thread_id": "1"}}
print("Executing SURE-Graph Pipeline...\n")

for event in app.stream(initial_state, thread):
    print(event)

current_state = app.get_state(thread)
print(current_state.values)

In [ ]:
print(current_state.values["current_summary"])

In [ ]:
from IPython.display import display, Markdown
display(Markdown(current_state.values["current_summary"]))

## Inference

In [ ]:
import time

def generate_summary(text: str):
    time.sleep(0.5)
    initial_state = {"source_text": text}
    config = {"configurable": {"thread_id": "1"}}
    try:
        response = app.invoke(initial_state, config)
    except Exception as e:
        response = ""
        print("Warning: Exception occured in app.invoke")
    return response

In [ ]:
sample_radiology_report = """
PA and lateral views of the chest provided. The heart size is within normal limits.
The mediastinal contours are unremarkable. The lungs are clear without focal consolidation,
pleural effusion, or pneumothorax. There are mild degenerative changes in the thoracic spine.
A 3mm nodule is noted in the right upper lobe, which is unchanged from the prior study dated one year ago.
"""

resp = generate_summary(sample_radiology_report)
print(resp["current_summary"])

In [ ]:
import pandas as pd

df = pd.read_json("https://huggingface.co/datasets/adasatwork-thesis/rrs/resolve/main/test.jsonl", lines=True)
df = df[:100]
df.info()

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()
df["agsumm"] = df["findings"].progress_map(lambda x: generate_summary(x)["current_summary"])

In [ ]:
df.to_json("rrs_test_agsumm.jsonl", orient="records", lines=True)